# COLMAP SfM Pipeline for 3D Gaussian Splatting

Based on the official [COLMAP Python example](https://github.com/colmap/colmap/blob/main/python/examples/example.py).

This notebook adapts the official COLMAP Python bindings (pycolmap) for use on Colab,
integrated with the arena-3dgs training pipeline.

**Pipeline:** Mount Drive → Load images → Downscale (optional) → COLMAP SfM → Validate → Train 3DGS

| Step | Cell | What | Est. Time | GPU? |
|------|------|------|-----------|------|
| 1 | Cell 1 | Mount Drive + set paths | 30s | No |
| 2 | Cell 2 | Install pycolmap-cuda12 + deps | 3–15 min | No |
| 3 | Cell 3 | Load images from Drive | ~1 min | No |
| 4 | Cell 4 | Downscale to 1080p + EXIF fix | ~1 min | No |
| 5 | Cell 5 | COLMAP feature extraction | 1–3 min | **T4** |
| 6 | Cell 6 | COLMAP feature matching | 1–5 min | **T4** |
| 7 | Cell 7 | COLMAP incremental SfM | 2–10 min | No |
| 8 | Cell 8 | Validate reconstruction | ~1 min | No |
| 9 | Cell 9 | Export text for 3DGS | ~1 min | No |
| 10 | Cell 10 | Quick test (3K iters) | ~7 min | **T4** |
| 11 | Cell 11 | Full training (30K iters) | ~30 min | **T4** |


In [ ]:
#@title === 1. Mount Google Drive + Set Paths ===
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, subprocess
from pathlib import Path

# --- EDIT THIS ---
DRIVE_PATH = "/content/drive/MyDrive/arena_3dgs"
# ---
os.makedirs(DRIVE_PATH, exist_ok=True)

WORK_DIR = Path("/content/gaussian-splatting")
WORK_DIR.mkdir(exist_ok=True)
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
SCRIPTS_DIR = Path("/content/scripts")
SCRIPTS_DIR.mkdir(exist_ok=True)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print(f"Drive: {DRIVE_PATH}")
print(f"Input: {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
#@title === 2. Install Dependencies (pycolmap-cuda12 + torch + gsplat) ===
DEPS_MARKER = os.path.join(DRIVE_PATH, ".colmap_deps_installed")
REPO_URL = "https://raw.githubusercontent.com/kaarthik-balakrishnan/arena-3dgs/main"

def pycolmap_works():
    try:
        import pycolmap
        return True
    except ImportError:
        return False

if os.path.exists(DEPS_MARKER) and pycolmap_works():
    print("Deps already installed.")
else:
    print("[1/4] Installing system deps...")
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx libglib2.0-0 2>/dev/null",
        shell=True,
    )

    print("[2/4] Installing pycolmap-cuda12...")
    subprocess.run("pip install pycolmap-cuda12 -q", shell=True)

    print("[3/4] Installing PyTorch + gsplat...")
    subprocess.run(
        "pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118 -q",
        shell=True,
    )
    subprocess.run(
        "pip install plyfile pillow opencv-python-headless tqdm gsplat scipy -q",
        shell=True,
    )

    print("[4/4] Downloading training scripts...")
    import urllib.request
    for name in ["train_3dgs_enhanced.py", "visualize_coverage.py"]:
        urllib.request.urlretrieve(
            f"{REPO_URL}/scripts/{name}",
            str(SCRIPTS_DIR / name),
        )

    Path(DEPS_MARKER).touch()
    print("\nAll dependencies ready!")

import pycolmap
print(f"PyCOLMAP version: {pycolmap.__version__}")

In [ ]:
#@title === 3. Load Images from Drive ===
# --- EDIT THIS: folder containing your images on Drive ---
SOURCE_PATH = os.path.join(DRIVE_PATH, "BrushTest/images")
# ---

if not os.path.isdir(SOURCE_PATH):
    raise NotADirectoryError(
        f"Source not found: {SOURCE_PATH}\n"
        f"Upload your images to a folder on Drive and update SOURCE_PATH.")

INPUT_DIR.mkdir(exist_ok=True)
count = 0
for fname in sorted(os.listdir(SOURCE_PATH)):
    if fname.lower().endswith((".jpg", ".jpeg", ".png")):
        shutil.copy2(
            os.path.join(SOURCE_PATH, fname),
            INPUT_DIR / fname,
        )
        count += 1

print(f"Loaded {count} images from {SOURCE_PATH}")
assert count > 0, "No images found!"

In [ ]:
#@title === 4. (Optional) Downscale to 1080p + Fix EXIF ===
# Run this if images have mixed resolutions or EXIF orientation issues.
# Creates a separate directory for processed images.
from PIL import Image

MAX_SIDE = 1080
COLMAP_IMAGE_DIR = WORK_DIR / "input_1080p"
COLMAP_IMAGE_DIR.mkdir(exist_ok=True)

count = 0
for fname in sorted(os.listdir(INPUT_DIR)):
    if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    img = Image.open(INPUT_DIR / fname)

    # Apply EXIF rotation
    try:
        orient = img.getexif().get(0x0112, 1)
        if orient == 6:
            img = img.rotate(-90, expand=True)
        elif orient == 8:
            img = img.rotate(90, expand=True)
        elif orient == 3:
            img = img.rotate(180, expand=True)
    except Exception:
        pass

    w, h = img.size
    if max(w, h) > MAX_SIDE:
        scale = MAX_SIDE / max(w, h)
        img = img.resize(
            (int(w * scale), int(h * scale)),
            Image.Resampling.LANCZOS,
        )

    img.save(COLMAP_IMAGE_DIR / fname, "JPEG", quality=95)
    count += 1

print(f"{count} images ready at {COLMAP_IMAGE_DIR}")
image_dir = COLMAP_IMAGE_DIR

# Also symlink for training (which expects input/images/)
TRAIN_IMAGES_DIR = INPUT_DIR / "images"
TRAIN_IMAGES_DIR.mkdir(exist_ok=True)
for fname in os.listdir(COLMAP_IMAGE_DIR):
    dst = TRAIN_IMAGES_DIR / fname
    if not dst.exists():
        dst.symlink_to(COLMAP_IMAGE_DIR / fname)
print(f"Training images dir: {TRAIN_IMAGES_DIR}")

In [ ]:
#@title === 5. COLMAP: Feature Extraction ===
# Uses the official pycolmap API with typed option objects (not dicts).
import torch
DB_PATH = WORK_DIR / "database.db"
SFM_PATH = WORK_DIR / "sparse"

if not DB_PATH.exists():
    print("Extracting features...")
    reader_opts = pycolmap.ImageReaderOptions(
        camera_model="SIMPLE_RADIAL", single_camera=True,
    )
    sift_opts = pycolmap.SiftExtractionOptions(max_num_features=8192)
    extract_opts = pycolmap.FeatureExtractionOptions(
        sift=sift_opts, use_gpu=torch.cuda.is_available(),
    )
    pycolmap.extract_features(
        DB_PATH, image_dir,
        reader_options=reader_opts,
        extraction_options=extract_opts,
    )
    print("Feature extraction complete.")
else:
    print("Database exists, skipping feature extraction.")

In [ ]:
#@title === 6. COLMAP: Feature Matching ===
# Sequential (overlap=20) + exhaustive matching.
# Sequential handles video-like sequences; exhaustive catches non-adjacent pairs.
import sqlite3

conn = sqlite3.connect(DB_PATH)
existing = conn.execute(
    "SELECT COUNT(*) FROM two_view_geometries"
).fetchone()[0]
conn.close()

if existing > 100:
    print(f"{existing} verified pairs already in DB. Skipping.")
else:
    seq_opts = pycolmap.SequentialPairingOptions(overlap=20)
    print("Sequential matching (overlap=20)...")
    pycolmap.match_sequential(DB_PATH, pairing_options=seq_opts)

    print("Exhaustive matching...")
    pycolmap.match_exhaustive(DB_PATH)

    conn = sqlite3.connect(DB_PATH)
    verified = conn.execute(
        "SELECT COUNT(*) FROM two_view_geometries"
    ).fetchone()[0]
    conn.close()
    print(f"Verified image pairs: {verified}")

In [ ]:
#@title === 7. COLMAP: Incremental SfM (Mapper) ===
# Uses the official pycolmap.incremental_mapping() with progress bar.
# Returns multiple reconstructions; picks the one with most registered images.
import enlighten

SFM_PATH.mkdir(exist_ok=True)

with pycolmap.Database.open(DB_PATH) as database:
    num_images = database.num_images()
    print(f"Database has {num_images} images")

if not (SFM_PATH / "images.bin").exists() and not (SFM_PATH / "images.txt").exists():
    print("Running incremental SfM...")
    with enlighten.Manager() as manager:
        pbar = manager.counter(total=num_images, desc="Images registered:")
        pbar.update(0, force=True)

        reconstructions = pycolmap.incremental_mapping(
            DB_PATH,
            image_dir,
            SFM_PATH,
            initial_image_pair_callback=lambda: pbar.update(2),
            next_image_callback=lambda: pbar.update(1),
        )

    if not reconstructions:
        raise RuntimeError("No reconstructions produced by COLMAP.")

    best = max(reconstructions.values(), key=lambda r: r.num_images())
    print(f"\nBest model: {best.summary()}")
    best.write(SFM_PATH)
else:
    print("Model exists, loading...")
    best = pycolmap.Reconstruction(SFM_PATH)
    print(best.summary())

In [ ]:
#@title === 8. Validate Reconstruction ===
import numpy as np

rec = pycolmap.Reconstruction(SFM_PATH)
n_img = rec.num_images()
n_pts = rec.num_points3D()
print(f"Registered images: {n_img}")
print(f"3D points:         {n_pts:,}")

if n_img == 0:
    raise RuntimeError("No images registered!")

# Camera centers
centers = []
for img in rec.images.values():
    cam_from_world = img.cam_from_world
    R = cam_from_world.rotation.matrix()
    t = cam_from_world.translation
    centers.append(-R.T @ t)
C = np.array(centers)
print(f"Camera span: X[{C[:,0].min():.1f},{C[:,0].max():.1f}] "
      f"Y[{C[:,1].min():.1f},{C[:,1].max():.1f}] "
      f"Z[{C[:,2].min():.1f},{C[:,2].max():.1f}]")

# Collinearity check
vec = C - C[0]
norms = np.linalg.norm(vec, axis=1)
valid = norms > 1e-6
if valid.sum() > 1:
    v0 = vec[valid][0] / norms[valid][0]
    dots = [
        np.dot(v / n, v0) for v, n in zip(vec[valid], norms[valid])
    ]
    mean_dot = np.mean(dots)
    status = "✓ good" if mean_dot < 0.85 else "⚠ colinear"
    print(f"Collinearity: {mean_dot:.3f} ({status})")

print("\n✓ COLMAP reconstruction validated.")

In [ ]:
#@title === 9. Export Text Format for 3DGS Training ===
# The training script expects cameras.txt, images.txt, points3D.txt
# in input/sparse/0/ directory.
SPARSE_TXT_DIR = INPUT_DIR / "sparse" / "0"
SPARSE_TXT_DIR.mkdir(parents=True, exist_ok=True)

rec.write_text(str(SPARSE_TXT_DIR))

for f in ["cameras.txt", "images.txt", "points3D.txt"]:
    p = SPARSE_TXT_DIR / f
    print(f"  {f}: {p.stat().st_size / 1024:.0f} KB")

# Save to Drive too
DRIVE_SPARSE = Path(DRIVE_PATH) / "sparse_3dgs"
DRIVE_SPARSE.mkdir(exist_ok=True)
for f in ["cameras.txt", "images.txt", "points3D.txt"]:
    shutil.copy2(SPARSE_TXT_DIR / f, DRIVE_SPARSE / f)
print(f"\nCopied to Drive: {DRIVE_SPARSE}")
print("\nReady for 3DGS training.")

In [ ]:
#@title === 10A: Quick Test (3000 iters, ~7 min) ===
TEST_ITERS = 3000
TEST_MAX_GAUSSIANS = 0
TEST_LOG_INTERVAL = 500
TEST_RANDOM_BG = False
TEST_OPACITY_RESET = 3000
TEST_DENSIFY_UNTIL = 1500
TEST_SH_DEGREE = 1
TEST_SH_DEGREE_INTERVAL = 500
TEST_FORCE_SPLIT_SCALE = 0.02
TEST_MAX_RES = 800

from scripts.colab_pipeline import train_3dgs
train_3dgs(
    input_dir=str(INPUT_DIR),
    iterations=TEST_ITERS,
    max_gaussians=TEST_MAX_GAUSSIANS,
    log_interval=TEST_LOG_INTERVAL,
    max_res=TEST_MAX_RES,
    output_name="quick_test",
    random_background=TEST_RANDOM_BG,
    opacity_reset_interval=TEST_OPACITY_RESET,
    densify_until_iter=TEST_DENSIFY_UNTIL,
    sh_degree=TEST_SH_DEGREE,
    sh_degree_interval=TEST_SH_DEGREE_INTERVAL,
    force_split_scale=TEST_FORCE_SPLIT_SCALE,
)

In [ ]:
#@title === 10B: Full Training (30000 iters, ~30 min) ===
FULL_ITERS = 30000
FULL_MAX_GAUSSIANS = 300000
FULL_LOG_INTERVAL = 1000
FULL_RANDOM_BG = False
FULL_OPACITY_RESET = 3000
FULL_DENSIFY_UNTIL = 15000
FULL_SH_DEGREE = 3
FULL_SH_DEGREE_INTERVAL = 1000
FULL_FORCE_SPLIT_SCALE = 0.02
FULL_MAX_RES = 800

from scripts.colab_pipeline import train_3dgs
train_3dgs(
    input_dir=str(INPUT_DIR),
    iterations=FULL_ITERS,
    max_gaussians=FULL_MAX_GAUSSIANS,
    log_interval=FULL_LOG_INTERVAL,
    max_res=FULL_MAX_RES,
    output_name="arena_3dgs",
    random_background=FULL_RANDOM_BG,
    opacity_reset_interval=FULL_OPACITY_RESET,
    densify_until_iter=FULL_DENSIFY_UNTIL,
    sh_degree=FULL_SH_DEGREE,
    sh_degree_interval=FULL_SH_DEGREE_INTERVAL,
    force_split_scale=FULL_FORCE_SPLIT_SCALE,
)

In [ ]:
#@title === 11. Export PLY + Copy to Drive ===
from plyfile import PlyData

PLY_SRC = os.path.join(str(OUTPUT_DIR), "arena_3dgs", "arena_3dgs.ply")
if os.path.exists(PLY_SRC):
    dst = "/content/arena_3dgs_pointcloud.ply"
    shutil.copy2(PLY_SRC, dst)
    size_mb = os.path.getsize(dst) / (1024 * 1024)

    ply = PlyData.read(dst)
    data = ply["vertex"].data
    print(f"PLY: {len(data):,} Gaussians, {size_mb:.1f} MB")

    required = [
        "x","y","z",
        "f_dc_0","f_dc_1","f_dc_2",
        "opacity","scale_0","scale_1","scale_2",
        "rot_0","rot_1","rot_2","rot_3",
    ]
    missing = [p for p in required if p not in data.dtype.names]
    print(f"Format: {'✓ valid' if not missing else '⚠ missing: ' + str(missing)}")

    drive_dst = os.path.join(DRIVE_PATH, os.path.basename(dst))
    shutil.copy2(dst, drive_dst)
    print(f"Saved to Drive: {drive_dst}")
else:
    print("No PLY found at expected path.")
    # Check quick test output
    quick_ply = os.path.join(str(OUTPUT_DIR), "quick_test", "arena_3dgs.ply")
    if os.path.exists(quick_ply):
        print(f"Quick test PLY available: {quick_ply}")